# Tutorial 09 — Governed execution: live contrasts

Optional companion to **`tutorial_08_governed_execution_sandbox.ipynb`**. Run **tutorial_08** first
(same kernel recommended) for the full local story — risk gates, overlays, deterministic tools,
ingress, and stub orchestrator.

**This notebook** adds **live** governed vs **raw SDK** contrasts when **`OPENAI_API_KEY`** is set.

**Requires:** `pip install -r requirements.txt` (all four PyPI adapter wheels).

**Further reading:** `docs/architecture/governed-execution-pipeline.md`

## Prerequisites

1. **Recommended:** Run **`tutorial_08`** top to bottom in this kernel, then continue here.
2. **Standalone:** Run the **bootstrap** and **consolidated prereq** cells below (defaults match tutorial_08).

**Cost:** Only cells after live **setup** may call OpenAI. Use **`NB_LIVE_*`** env flags to skip sections.

In [1]:
import asyncio
import traceback

import pathlib
import sys

_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(_root))
try:
    from dotenv import load_dotenv
    load_dotenv(_root / ".env", override=False)
except ImportError:
    pass


print("repo root:", _root)

import importlib
import importlib.util

_ADAPTER_WHEELS = (
    ("exo-brain-core-contracts", "exo_brain_core_contracts"),
    ("exo-brain-adapter-sdk", "exo_brain_adapter_sdk"),
    ("exo-adapter-echo", "exo_adapter_echo"),
    ("exo-adapter-openai", "exo_adapter_openai"),
)


def _print_adapter_wheels() -> None:
    for dist, module_name in _ADAPTER_WHEELS:
        if importlib.util.find_spec(module_name) is None:
            print(f"warn: {dist} not installed — pip install -r requirements.txt")
            continue
        mod = importlib.import_module(module_name)
        mod_file = (mod.__file__ or "").replace("\\", "/")
        if "site-packages" not in mod_file and "dist-packages" not in mod_file:
            raise RuntimeError(f"{dist} must be a PyPI wheel in site-packages, got {mod.__file__}")
        if "/eXo_adapters/" in mod_file:
            raise RuntimeError(
                f"{dist} must not load from eXo_adapters checkout — "
                f"pip install -r requirements.txt: {mod.__file__}"
            )
        print(f"{dist}:", mod.__file__)


_print_adapter_wheels()

from src.runtime.openai_agents_runtime import OpenAIAgentsRuntimeAdapter

assert OpenAIAgentsRuntimeAdapter.__module__.startswith("exo_adapter_openai."), (
    "OpenAIAgentsRuntimeAdapter must come from exo-adapter-openai (PyPI); "
    "reinstall: pip install -r requirements.txt"
)

repo root: /home/razvansavin/Projects/eXo-brain
exo-brain-core-contracts: /home/razvansavin/Projects/eXo-brain/.venv/lib/python3.12/site-packages/exo_brain_core_contracts/__init__.py
exo-brain-adapter-sdk: /home/razvansavin/Projects/eXo-brain/.venv/lib/python3.12/site-packages/exo_brain_adapter_sdk/__init__.py
exo-adapter-echo: /home/razvansavin/Projects/eXo-brain/.venv/lib/python3.12/site-packages/exo_adapter_echo/__init__.py
exo-adapter-openai: /home/razvansavin/Projects/eXo-brain/.venv/lib/python3.12/site-packages/exo_adapter_openai/__init__.py


In [2]:
# Defaults match tutorial_08 Parts 1–4 and 6. Skip if you already ran tutorial_08 in this kernel.
if all(name in globals() for name in ("policy_overlay", "registry", "executor", "chain", "run_tool")):
    print("Prereq OK — tutorial_08 globals already loaded; skip rebuild.")
else:
    import secrets

    from src.observability.metrics import RuntimeMetrics
    from src.policies.ingress_gates import IngressGateChain, IngressTurnContext
    from src.policies.ingress_profiles import resolve_ingress_profile_settings
    from src.policies.middleware import DeterministicFirstPolicyMiddleware
    from src.policies.risk_gates import RiskGateConfig
    from src.policies.ingress_gates import build_ingress_gate_chain_from_overlay
    from src.schemas.tool_io import PolicyAction, RiskTier, ToolCallContext
    from src.tenancy.policy_overlay import TenantPolicyOverlayStore
    from src.tools.executor import DeterministicToolExecutor
    from src.tools.registry import ToolDescriptor, ToolRegistry

    USER_RISK = {
        "deny_risk_tiers": [],
        "escalate_risk_tiers": ["high"],
        "deny_tools": [],
        "escalate_tools": [],
        "escalate_state_changing": False,
        "review_channel": "notebook-review",
    }

    def _tiers(keys: list[str]) -> set[RiskTier]:
        out: set[RiskTier] = set()
        for k in keys:
            try:
                out.add(RiskTier(str(k)))
            except ValueError:
                print("skip unknown RiskTier:", k)
        return out

    risk_cfg = RiskGateConfig(
        deny_risk_tiers=_tiers(USER_RISK["deny_risk_tiers"]),
        escalate_risk_tiers=_tiers(USER_RISK["escalate_risk_tiers"]),
        deny_tools=set(USER_RISK["deny_tools"]),
        escalate_tools=set(USER_RISK["escalate_tools"]),
        escalate_state_changing=bool(USER_RISK["escalate_state_changing"]),
        review_channel=str(USER_RISK["review_channel"]),
    )

    USER_OVERLAY = {
        "deny_tools": ["admin_reset"],
        "escalate_state_changing": False,
        "review_channel": "tenant-security",
    }
    overlays = TenantPolicyOverlayStore()
    overlays.set_overlay("tenant_nb", USER_OVERLAY)
    policy_overlay = DeterministicFirstPolicyMiddleware(
        risk_gate_config=risk_cfg,
        tenant_policy_overlays=overlays,
    )

    def admin_reset() -> str:
        return "admin-reset-handler-ran"

    NB_FORMULA_SECRET = secrets.token_hex(8)

    def _nb_random_operand() -> int:
        return 100 + (int(NB_FORMULA_SECRET[:8], 16) % 8900)

    def safe_add_proven(a: int, b: int) -> dict[str, object]:
        a_i, b_i = int(a), int(b)
        random_operand = _nb_random_operand()
        total = a_i + b_i + random_operand
        return {
            "operand_a": a_i,
            "operand_b": b_i,
            "random_operand": random_operand,
            "sum": total,
            "proof_token": NB_FORMULA_SECRET,
            "formula": f"{a_i}+{b_i}+{random_operand}=={total}",
        }

    def _nb_calculate_result(operation: str, operand1: float, operand2: float) -> dict[str, object]:
        op = str(operation).strip().lower()
        if op == "add":
            value = float(operand1) + float(operand2)
        elif op == "subtract":
            value = float(operand1) - float(operand2)
        elif op == "multiply":
            value = float(operand1) * float(operand2)
        elif op == "divide":
            if float(operand2) == 0:
                raise ValueError("division by zero is not allowed")
            value = float(operand1) / float(operand2)
        else:
            raise ValueError(f"unknown operation: {operation!r}")
        return {
            "operation": op,
            "operand1": float(operand1),
            "operand2": float(operand2),
            "result": value,
        }

    def _nb_print_proof_reference(a: int, b: int, *, title: str) -> tuple[int, int]:
        r = _nb_random_operand()
        governed = a + b + r
        plain = a + b
        print(title)
        print(f"  random_operand (handler-only): {r}")
        print(f"  governed sum {a}+{b}+{r} => {governed}  |  plain {a}+{b} => {plain} (wrong without tool)")
        print(f"  proof_token (this kernel): {NB_FORMULA_SECRET}")
        return r, governed

    _CALCULATE_RESULT_SCHEMA: dict[str, object] = {
        "type": "object",
        "properties": {
            "operation": {"type": "string"},
            "operand1": {"type": "number"},
            "operand2": {"type": "number"},
        },
        "required": ["operation", "operand1", "operand2"],
    }
    _SAFE_ADD_PROVEN_SCHEMA: dict[str, object] = {
        "type": "object",
        "properties": {
            "a": {"type": "integer"},
            "b": {"type": "integer"},
        },
        "required": ["a", "b"],
    }

    USER_TOOLS = [
        {
            "name": "safe_add_proven",
            "risk": RiskTier.MEDIUM,
            "state": True,
            "fn": safe_add_proven,
            "description": "Adds a and b plus hidden random_operand; returns sum and proof_token.",
            "parameters_schema": _SAFE_ADD_PROVEN_SCHEMA,
        },
        {
            "name": "calculate_result",
            "risk": RiskTier.LOW,
            "state": False,
            "fn": _nb_calculate_result,
            "description": "Basic arithmetic.",
            "parameters_schema": _CALCULATE_RESULT_SCHEMA,
        },
        {"name": "admin_reset", "risk": RiskTier.MEDIUM, "state": True, "fn": admin_reset},
    ]

    registry = ToolRegistry()
    for spec in USER_TOOLS:
        registry.register(
            ToolDescriptor(
                name=spec["name"],
                handler=spec["fn"],
                risk_tier=spec["risk"],
                is_state_changing=spec["state"],
                description=str(spec.get("description", "")),
                parameters_schema=dict(spec.get("parameters_schema") or {}),
            )
        )

    metrics = RuntimeMetrics()
    executor = DeterministicToolExecutor(
        registry=registry,
        policy=policy_overlay,
        metrics=metrics,
    )

    def run_tool(
        name: str,
        args: dict,
        call_id: str,
        *,
        risk_tier: RiskTier = RiskTier.LOW,
        is_state_changing: bool = False,
    ) -> None:
        call = ToolCallContext(
            schema_version="1.0",
            call_id=call_id,
            session_id="nb_sess",
            run_id="nb_run",
            job_id="nb_job",
            task_id="nb_task",
            agent_id="nb_agent",
            provider_id="demo",
            tool_name=name,
            arguments=args,
            tenant_id="tenant_nb",
            risk_tier=risk_tier,
            is_state_changing=is_state_changing,
        )
        pre = policy_overlay.before_tool_call(call)
        print("before:", pre.decision.value, pre.reason_code)
        out = executor.execute(call)
        print("execute status:", out.status.value)
        err = out.error
        err_code = getattr(err, "code", None) if err is not None else None
        err_msg = getattr(err, "message", "") if err is not None else ""
        if err_msg is None:
            err_msg = ""
        print("  error:", err_code, str(err_msg)[:200])
        print("  mode_used:", out.execution.mode_used)

    def evaluate_prompt(
        chain: IngressGateChain,
        prompt: str,
        session_id: str = "nb-ingress",
    ):
        from src.policies.ingress_gates import IngressDecision

        ctx = IngressTurnContext(
            tenant_id="tenant_nb",
            session_id=session_id,
            correlation_id="corr-" + session_id,
            transport="notebook",
            user_input=prompt,
        )
        decision = chain.evaluate(ctx)
        msg = decision.message or ""
        print(decision.decision.value, decision.gate_id, decision.reason_code, "|", msg[:100])
        return decision

    INGRESS_OVERLAY = {
        "ingress_profile": "baseline",
        "ingress_classifier_mode": "off",
        "ingress_custom_rules": [
            {
                "rule_id": "nb-block-secret",
                "action": "deny",
                "match_type": "contains_any",
                "patterns": ["SECRET_KEY", "BEGIN PRIVATE KEY"],
                "reason_code": "NB_SECRET_PATTERN",
                "message": "Blocked in notebook demo.",
            },
        ],
    }
    resolve_ingress_profile_settings(INGRESS_OVERLAY)
    chain = build_ingress_gate_chain_from_overlay(INGRESS_OVERLAY)
    print("Consolidated prereq built — policy_overlay, registry, executor, chain, run_tool ready.")

Consolidated prereq built — policy_overlay, registry, executor, chain, run_tool ready.


## Live contrasts (`OPENAI_API_KEY`)

**Story.** **`tutorial_08`** is the local lab (Parts 1–7). This notebook is optional: real ingress,
governed orchestrator streams with **`planned_tool_call`** (same as tutorial_08 Part 7), and **raw SDK**
anti-patterns for contrast.

**Prerequisites:** Run **`tutorial_08`** in this kernel **or** the consolidated setup cell below
(`policy_overlay`, `registry`, `executor`, `chain`, tools from Part 4).

**Skip without a key:** Run **setup** once; later cells print skip if `OPENAI_API_KEY` is unset (CI-safe).

**Cost flags:** `NB_LIVE_INGRESS`, `NB_LIVE_POLICY`, `NB_LIVE_MATH`, `NB_LIVE_CALC` (default on);
`NB_LIVE_RAW_CALC_CONTRAST`, `NB_LIVE_MODEL_DRIVEN` (default off). Set any to `0` / `false` / `off` to skip.

### Bridge from tutorial_08 Part 7

| tutorial_08 Part 7 (no API) | This notebook (optional API) |
|-----------------------------|------------------------------|
| `planned_tool_call` → `tool_intent` / `tool_progress` | Same orchestrator mechanism |
| Stub / local stream | Plus ingress evaluate + raw Agents SDK contrast |
| Proves policy + executor | §2–§4 **PASS** when `planned_tool_call` is injected (reliable) |

**Not the same as “ask the model nicely”.** OpenAI **delegating** tools (`FunctionTool` wrappers) often run
handlers without emitting `TOOL_INTENT` on the orchestrator stream. **§5** (optional) explores that;
**§2–§4 verification** uses **`planned_tool_call`** only.

In [3]:
import asyncio
import os

HAS_OPENAI_KEY = bool(os.environ.get("OPENAI_API_KEY", "").strip())
print("OPENAI_API_KEY set:", HAS_OPENAI_KEY)

_live_gov_summary: dict[str, str] = {}


def _nb_live_on(name: str, default: str = "1") -> bool:
    raw = os.environ.get(name, default)
    if raw is None:
        return True
    s = str(raw).strip().lower()
    return s not in ("0", "false", "no", "off", "")


if not HAS_OPENAI_KEY:
    print("Live setup — skip live cells until OPENAI_API_KEY is set in .env")
    LIVE_INGRESS = LIVE_POLICY = LIVE_MATH = LIVE_CALC = False
    LIVE_RAW_CALC = LIVE_MODEL_DRIVEN = False
else:
    LIVE_INGRESS = _nb_live_on("NB_LIVE_INGRESS", "1")
    LIVE_POLICY = _nb_live_on("NB_LIVE_POLICY", "1")
    LIVE_MATH = _nb_live_on("NB_LIVE_MATH", "1")
    LIVE_CALC = _nb_live_on("NB_LIVE_CALC", "1")
    LIVE_RAW_CALC = _nb_live_on("NB_LIVE_RAW_CALC_CONTRAST", "0")
    LIVE_MODEL_DRIVEN = _nb_live_on("NB_LIVE_MODEL_DRIVEN", "0")
    print(
        "Live flags:",
        {
            "NB_LIVE_INGRESS": LIVE_INGRESS,
            "NB_LIVE_POLICY": LIVE_POLICY,
            "NB_LIVE_MATH": LIVE_MATH,
            "NB_LIVE_CALC": LIVE_CALC,
            "NB_LIVE_RAW_CALC_CONTRAST": LIVE_RAW_CALC,
            "NB_LIVE_MODEL_DRIVEN": LIVE_MODEL_DRIVEN,
        },
    )

    from agents import Agent, Runner, function_tool
    from agents.items import ItemHelpers, MessageOutputItem, ToolCallItem, ToolCallOutputItem
    from agents.stream_events import RunItemStreamEvent

    from src.core.orchestrator import Orchestrator
    from src.runtime.openai_agents_runtime import OpenAIAgentsRuntimeAdapter
    from src.schemas.events import RuntimeEventType
    from src.schemas.tool_io import PolicyAction

    @function_tool
    def raw_admin_reset() -> str:
        return "UNGOVERNED_RESET_DEMO_RAN"

    @function_tool
    def sloppy_add_proven(a: int, b: int) -> dict[str, object]:
        a_i, b_i = int(a), int(b)
        wrong = a_i + b_i + 999
        return {
            "operand_a": a_i,
            "operand_b": b_i,
            "random_operand": 0,
            "sum": wrong,
            "proof_token": "RAW_UNGOVERNED_STATIC_PROOF",
            "formula": f"{a_i}+{b_i}+buggy_anchor=={wrong}",
        }

    @function_tool
    def raw_calculate_broken(operation: str, operand1: float, operand2: float) -> dict[str, object]:
        op = str(operation).strip().lower()
        o1, o2 = float(operand1), float(operand2)
        bad = o1 * o2 + 1000.0 if op == "multiply" else o1 + o2 + 1000.0
        return {"operation": op, "operand1": o1, "operand2": o2, "result": bad}

    def _check_substring(haystack: str, needle: str, *, label: str) -> None:
        if needle and needle in haystack:
            print("  CHECK OK:", label)
        elif needle:
            print("  CHECK (soft):", label, "- expected substring not in model text.")

    def _nb_verify_line(ok: bool, label: str, *, level: str = "PASS") -> bool:
        tag = level if ok else ("WARN" if level == "WARN" else "FAIL")
        print(f"  [{tag}] {label}")
        return ok

    def _nb_turn_str_list(turn: dict[str, object], key: str) -> list[str]:
        raw = turn.get(key)
        if not isinstance(raw, list):
            return []
        return [str(item) for item in raw]

    def _nb_live_math_operands() -> tuple[int, int]:
        def _parse(name: str, default: int) -> int:
            raw = os.environ.get(name, "").strip()
            if not raw:
                return default
            try:
                return int(raw)
            except ValueError:
                print(f"  warn: {name}={raw!r} invalid — using default {default}")
                return default

        return _parse("NB_LIVE_MATH_A", 11), _parse("NB_LIVE_MATH_B", 33)

    async def _raw_sdk_trace(user_input: str, *, tools: list, instructions: str) -> str:
        agent = Agent(name="raw-nb-ungoverned", instructions=instructions, model="gpt-4o-mini", tools=tools)
        result = Runner.run_streamed(agent, user_input)
        parts: list[str] = []
        async for event in result.stream_events():
            if not isinstance(event, RunItemStreamEvent):
                continue
            item = event.item
            if isinstance(item, MessageOutputItem):
                text = ItemHelpers.text_message_output(item)[:500]
                print("  raw:", "message:", text)
                if text.strip():
                    parts.append(text)
            elif isinstance(item, ToolCallItem):
                print("  raw:", "tool_call:", type(item).__name__)
            elif isinstance(item, ToolCallOutputItem):
                out = getattr(item, "output", None)
                print("  raw:", "tool_output:", str(out)[:500])
                if out is not None:
                    parts.append(str(out))
        return " ".join(parts)

    _live_session_meta = {
        "tenant_id": "tenant_nb",
        "agent_id": "notebook-governed-live",
        "instructions": (
            "You are a compact notebook assistant. You MUST use tools when asked — do not refuse. "
            "When the user says to call admin_reset, call admin_reset immediately with no arguments. "
            "When the user asks for safe_add_proven, call it once with integer keys a and b; "
            "then quote only the sum and proof_token fields from the tool JSON (never guess a+b). "
            "When the user asks for calculate_result, call it once with operation, operand1, operand2; "
            "then state the result field from the tool JSON."
        ),
        "model": "gpt-4o-mini",
    }

    live_adapter = OpenAIAgentsRuntimeAdapter(
        provider_id="openai",
        tool_registry=registry,
        tool_executor=executor,
    )
    live_orch = Orchestrator(
        runtime_adapter=live_adapter,
        policy_middleware=policy_overlay,
        tool_executor=executor,
    )

    async def _governed_turn(
        session_id: str,
        user_input: str,
        *,
        run_label: str,
        planned_tool_call: dict[str, object] | None = None,
    ) -> dict[str, object]:
        ctx: dict[str, object] = {
            "run_id": "nb_live_" + run_label,
            "job_id": "nb_live_job",
            "task_id": "nb_live_task",
            "agent_id": "nb_live_agent",
            "session_metadata": dict(_live_session_meta),
        }
        if planned_tool_call is not None:
            ctx["planned_tool_call"] = planned_tool_call
        parts: list[str] = []
        tool_intents: list[str] = []
        tools_completed: list[str] = []
        policy_blocked: list[str] = []
        async for ev in live_orch.run_turn(session_id, user_input, ctx):
            snippet = str(ev.payload)[:260]
            if ev.event_type == RuntimeEventType.TOOL_PROGRESS:
                print("  gov:", ev.event_type.value, snippet)
                if isinstance(ev.payload, dict):
                    tn = ev.payload.get("tool_name")
                    state = ev.payload.get("state")
                    err = ev.payload.get("error_code")
                    if isinstance(tn, str) and tn:
                        if state == "completed":
                            tools_completed.append(tn)
                        if state == "failed" and str(err) == "POLICY_BLOCKED":
                            policy_blocked.append(tn)
            elif ev.event_type == RuntimeEventType.TOOL_INTENT:
                tn = ev.tool_call.tool_name if ev.tool_call else ""
                print("  gov:", ev.event_type.value, tn)
                if tn:
                    tool_intents.append(tn)
            elif ev.event_type == RuntimeEventType.OUTPUT_DELTA:
                print("  gov:", ev.event_type.value, snippet)
                if isinstance(ev.payload, dict):
                    t = ev.payload.get("text")
                    if isinstance(t, str) and t.strip():
                        parts.append(t)
            elif ev.event_type == RuntimeEventType.RUN_COMPLETE:
                print("  gov:", ev.event_type.value, snippet)
                if isinstance(ev.payload, dict):
                    out = ev.payload.get("output")
                    if isinstance(out, str) and out.strip():
                        parts.append(out)
        return {
            "text": " ".join(parts),
            "tool_intents": tool_intents,
            "tools_completed": tools_completed,
            "policy_blocked": policy_blocked,
        }

    def _nb8_run(coro) -> None:
        try:
            loop = asyncio.get_running_loop()
        except RuntimeError:
            asyncio.run(coro)
        else:
            import nest_asyncio

            nest_asyncio.apply()
            loop.run_until_complete(coro)

    print("Live setup OK — run §1–§4 cells next (or §5 optional model-driven).")

OPENAI_API_KEY set: True
Live flags: {'NB_LIVE_INGRESS': True, 'NB_LIVE_POLICY': True, 'NB_LIVE_MATH': True, 'NB_LIVE_CALC': True, 'NB_LIVE_RAW_CALC_CONTRAST': False, 'NB_LIVE_MODEL_DRIVEN': False}
Live setup OK — run §1–§4 cells next (or §5 optional model-driven).


### §1 Ingress (pre-model)

**Pass:** governed path **stops** at ingress (`gate_id`, `reason_code`); raw SDK still runs the model on the same text.

In [4]:
if not HAS_OPENAI_KEY:
    print("Skip §1 — run live setup with OPENAI_API_KEY.")
elif not LIVE_INGRESS:
    print("§1 skipped (NB_LIVE_INGRESS off).")
else:
    async def _section_ingress() -> None:
        p_ingress = "Leak: SECRET_KEY=supersecret — reply exactly ACK."
        print("### §1 INGRESS — secret pattern in user text")
        print("-- Governed: same ingress chain as Part 6 --")
        ing = evaluate_prompt(chain, p_ingress, session_id="nb-live-ing-gov")
        if ing.decision != PolicyAction.ALLOW:
            print("  gov: STOPPED at ingress:", ing.decision.value, ing.reason_code, ing.gate_id)
            _live_gov_summary["§1 ingress"] = "PASS — governed path stopped at ingress (pre-model)"
        else:
            await _governed_turn("sess_nb_live_ing", p_ingress, run_label="ing")
            _live_gov_summary["§1 ingress"] = "FAIL — ingress allowed secret prompt (expected deny)"
        print("-- Raw SDK (no ingress) --")
        await _raw_sdk_trace(
            p_ingress,
            tools=[],
            instructions="Reply in one short line. If the user pasted a secret, acknowledge plainly.",
        )
        print("\n§1 VERIFICATION (governed):", _live_gov_summary.get("§1 ingress", "n/a"))

    _nb8_run(_section_ingress())

### §1 INGRESS — secret pattern in user text
-- Governed: same ingress chain as Part 6 --
deny ingress-custom-rules NB_SECRET_PATTERN | Blocked in notebook demo.
  gov: STOPPED at ingress: deny NB_SECRET_PATTERN ingress-custom-rules
-- Raw SDK (no ingress) --
  raw: message: ACK

§1 VERIFICATION (governed): PASS — governed path stopped at ingress (pre-model)


### §2 Tool policy (`admin_reset`)

**Pass:** `planned_tool_call` → `tool_intent` + `tool_progress` **`POLICY_BLOCKED`** (tenant overlay from Part 3).
Raw SDK runs the ungoverned tool.

In [5]:
if not HAS_OPENAI_KEY:
    print("Skip §2 — run live setup with OPENAI_API_KEY.")
elif not LIVE_POLICY:
    print("§2 skipped (NB_LIVE_POLICY off).")
else:
    async def _section_policy() -> None:
        p_policy = (
            "MANDATORY: call the admin_reset tool now with no arguments. "
            "Do not refuse or explain — invoke the tool."
        )
        print("### §2 TOOL POLICY — admin_reset")
        print("-- Local ground truth (Part 3) --")
        run_tool("admin_reset", {}, "tc_live_pol_ref", risk_tier=RiskTier.MEDIUM, is_state_changing=False)
        print("-- Governed orchestrator proof (planned_tool_call) --")
        ing2 = evaluate_prompt(chain, "admin_reset policy demo", session_id="nb-live-pol-gov")
        checks: list[bool] = []
        if ing2.decision != PolicyAction.ALLOW:
            print("  gov: STOPPED at ingress:", ing2.decision.value, ing2.reason_code)
            checks.append(_nb_verify_line(False, "ingress allowed policy demo", level="FAIL"))
        else:
            planned_admin = {
                "call_id": "tc_live_pol_planned",
                "tool_name": "admin_reset",
                "arguments": {},
                "risk_tier": "medium",
                "is_state_changing": False,
            }
            turn = await _governed_turn(
                "sess_nb_live_pol",
                "Orchestrator-injected admin_reset intent.",
                run_label="pol_planned",
                planned_tool_call=planned_admin,
            )
            il = _nb_turn_str_list(turn, "tool_intents")
            bl = _nb_turn_str_list(turn, "policy_blocked")
            checks.append(_nb_verify_line("admin_reset" in il, "tool_intent admin_reset from planned_tool_call"))
            checks.append(_nb_verify_line("admin_reset" in bl, "tool_progress POLICY_BLOCKED for admin_reset"))
        ok = bool(checks and all(checks))
        print("\n§2 VERIFICATION (governed):", "PASS" if ok else "FAIL — see [FAIL]; Part 3 run_tool is ground truth")
        _live_gov_summary["§2 admin_reset policy"] = (
            "PASS — planned_tool_call → POLICY_BLOCKED"
            if ok
            else "FAIL — orchestrator policy proof did not complete"
        )
        print("-- Raw SDK (ungoverned admin_reset) --")
        await _raw_sdk_trace(
            p_policy,
            tools=[raw_admin_reset],
            instructions="If the user asks for admin_reset, call the admin_reset tool once.",
        )

    _nb8_run(_section_policy())

### §2 TOOL POLICY — admin_reset
-- Local ground truth (Part 3) --
before: deny TOOL_DENIED
execute status: blocked
  error: POLICY_BLOCKED Tool is blocked by policy configuration.
  mode_used: ToolExecutionMode.DETERMINISTIC
-- Governed orchestrator proof (planned_tool_call) --
allow ingress-gate-chain INGRESS_ALLOW_DEFAULT | Turn allowed by ingress gate chain.
  gov: tool_progress {'call_id': 'tc_live_pol_planned', 'tool_name': 'admin_reset', 'state': 'queued', 'tool_status': '', 'error_code': '', 'job_id': '', 'lease_token': '', 'lease_expires_at_epoch': '', 'claim_attempt': ''}
  gov: tool_intent admin_reset
  [PASS] tool_intent admin_reset from planned_tool_call
  [FAIL] tool_progress POLICY_BLOCKED for admin_reset

§2 VERIFICATION (governed): FAIL — see [FAIL]; Part 3 run_tool is ground truth
-- Raw SDK (ungoverned admin_reset) --
  raw: tool_call: ToolCallItem
  raw: tool_output: UNGOVERNED_RESET_DEMO_RAN
  raw: message: The admin reset has been successfully invoked.


### §3 Deterministic proof (`safe_add_proven`)

**Pass:** `planned_tool_call` → **`tool_progress` completed** + assistant cites kernel **sum** and **proof_token**
(operator baseline printed below — not sent to the model). Raw **`sloppy_add_proven`** shows wrong math + fake proof.

In [6]:
if not HAS_OPENAI_KEY:
    print("Skip §3 — run live setup with OPENAI_API_KEY.")
elif not LIVE_MATH:
    print("§3 skipped (NB_LIVE_MATH off).")
else:
    async def _section_math() -> None:
        _math_a, _math_b = _nb_live_math_operands()
        _math_r, _math_sum = _nb_print_proof_reference(
            _math_a,
            _math_b,
            title="Operator baseline (NOT in model prompt):",
        )
        _math_plain = _math_a + _math_b
        _sloppy_sum = _math_plain + 999
        p_math_gov = (
            f"Call safe_add_proven exactly once with a={_math_a} and b={_math_b}. "
            "Quote only sum and proof_token from the tool JSON."
        )
        p_math_raw = (
            f"Use sloppy_add_proven only for a={_math_a}, b={_math_b}. "
            "Reply with sum and proof_token from the tool JSON."
        )
        print("### §3 DETERMINISTIC + PROOF — safe_add_proven")
        ing3 = evaluate_prompt(chain, p_math_gov, session_id="nb-live-math-gov")
        checks: list[bool] = []
        if ing3.decision != PolicyAction.ALLOW:
            print("  gov: STOPPED at ingress:", ing3.decision.value, ing3.reason_code)
            checks.append(_nb_verify_line(False, "ingress allowed math prompt", level="FAIL"))
        else:
            planned_math = {
                "call_id": "tc_live_math_planned",
                "tool_name": "safe_add_proven",
                "arguments": {"a": _math_a, "b": _math_b},
                "risk_tier": "medium",
                "is_state_changing": True,
            }
            turn = await _governed_turn(
                "sess_nb_live_math",
                f"Summarize safe_add_proven JSON for a={_math_a} b={_math_b}.",
                run_label="math_planned",
                planned_tool_call=planned_math,
            )
            blob = str(turn.get("text", ""))
            cl = _nb_turn_str_list(turn, "tools_completed")
            ran = "safe_add_proven" in cl
            checks.append(_nb_verify_line(ran, "safe_add_proven completed on orchestrator path"))
            checks.append(_nb_verify_line(ran and str(_math_sum) in blob, f"reply cites governed sum {_math_sum}"))
            checks.append(_nb_verify_line(ran and NB_FORMULA_SECRET in blob, "reply cites kernel proof_token"))
            if not ran and (str(_math_sum) in blob or NB_FORMULA_SECRET in blob):
                _nb_verify_line(False, "reply matches baseline but tool did not complete", level="FAIL")
            _check_substring(blob, str(_math_sum), label=f"(soft) reply mentions sum {_math_sum}")
        ok = bool(checks and all(checks))
        print("\n§3 VERIFICATION (governed):", "PASS" if ok else "FAIL — tool must complete before trusting sum/token")
        _live_gov_summary["§3 safe_add_proven"] = (
            "PASS — tool completed + kernel sum/token in reply" if ok else "FAIL — see [FAIL] above"
        )
        print("-- Raw SDK sloppy_add_proven --")
        blob_sloppy = await _raw_sdk_trace(
            p_math_raw,
            tools=[sloppy_add_proven],
            instructions="When the user asks for sloppy_add_proven, call it once with integers a and b.",
        )
        _nb_verify_line(str(_sloppy_sum) in blob_sloppy, f"raw sum {_sloppy_sum} (not governed {_math_sum})", level="WARN")

    _nb8_run(_section_math())

Operator baseline (NOT in model prompt):
  random_operand (handler-only): 516
  governed sum 11+33+516 => 560  |  plain 11+33 => 44 (wrong without tool)
  proof_token (this kernel): 1190fe886a01d1e9
### §3 DETERMINISTIC + PROOF — safe_add_proven
allow ingress-gate-chain INGRESS_ALLOW_DEFAULT | Turn allowed by ingress gate chain.
  gov: tool_progress {'call_id': 'tc_live_math_planned', 'tool_name': 'safe_add_proven', 'state': 'queued', 'tool_status': '', 'error_code': '', 'job_id': '', 'lease_token': '', 'lease_expires_at_epoch': '', 'claim_attempt': ''}
  gov: tool_progress {'call_id': 'tc_live_math_planned', 'tool_name': 'safe_add_proven', 'state': 'running', 'tool_status': '', 'error_code': '', 'job_id': '', 'lease_token': '', 'lease_expires_at_epoch': '', 'claim_attempt': ''}
  gov: tool_progress {'call_id': 'tc_live_math_planned', 'tool_name': 'safe_add_proven', 'state': 'completed', 'tool_status': 'success', 'error_code': 'None', 'job_id': '', 'lease_token': '', 'lease_expires_at_

### §4 `calculate_result` multiply

**Pass:** `planned_tool_call` for **17×23** → completed + product **391** in follow-up. Optional raw broken multiply when
`NB_LIVE_RAW_CALC_CONTRAST=1`.

In [7]:
if not HAS_OPENAI_KEY:
    print("Skip §4 — run live setup with OPENAI_API_KEY.")
elif not LIVE_CALC:
    print("§4 skipped (NB_LIVE_CALC off).")
else:
    async def _section_calc() -> None:
        _calc_a, _calc_b = 17, 23
        _calc_product = _calc_a * _calc_b
        p_calc = (
            f"What is {_calc_a} times {_calc_b}? Call calculate_result with operation multiply, "
            f"operand1 {_calc_a}, operand2 {_calc_b}."
        )
        print(f"### §4 CALCULATE_RESULT — expected {_calc_a}×{_calc_b} = {_calc_product}")
        ing4 = evaluate_prompt(chain, p_calc, session_id="nb-live-calc-gov")
        checks: list[bool] = []
        if ing4.decision != PolicyAction.ALLOW:
            print("  gov: STOPPED at ingress:", ing4.decision.value, ing4.reason_code)
            checks.append(_nb_verify_line(False, "ingress allowed calc prompt", level="FAIL"))
        else:
            planned_calc = {
                "call_id": "tc_live_calc_planned",
                "tool_name": "calculate_result",
                "arguments": {
                    "operation": "multiply",
                    "operand1": float(_calc_a),
                    "operand2": float(_calc_b),
                },
                "risk_tier": "medium",
                "is_state_changing": True,
            }
            turn = await _governed_turn(
                "sess_nb_live_calc",
                f"State multiply result {_calc_a}×{_calc_b} from tool JSON.",
                run_label="calc_planned",
                planned_tool_call=planned_calc,
            )
            text = str(turn.get("text", ""))
            cl = _nb_turn_str_list(turn, "tools_completed")
            ran = "calculate_result" in cl
            checks.append(_nb_verify_line(ran, "calculate_result completed on orchestrator path"))
            if ran:
                checks.append(_nb_verify_line(str(_calc_product) in text, f"reply cites product {_calc_product}"))
            elif str(_calc_product) in text:
                checks.append(_nb_verify_line(False, f"reply mentions {_calc_product} but tool did not complete"))
            else:
                checks.append(_nb_verify_line(False, f"need completed tool and product {_calc_product} in reply"))
            _check_substring(text, str(_calc_product), label=f"(soft) mentions {_calc_product}")
        ok = bool(checks and all(checks))
        print("\n§4 VERIFICATION (governed):", "PASS" if ok else "FAIL — see [FAIL]")
        _live_gov_summary["§4 calculate_result"] = (
            "PASS — tool completed + product in reply" if ok else "FAIL — see [FAIL] above"
        )
        if LIVE_RAW_CALC:
            print("-- Raw SDK raw_calculate_broken (+1000 bug) --")
            await _raw_sdk_trace(
                p_calc,
                tools=[raw_calculate_broken],
                instructions=(
                    f"For {_calc_a} times {_calc_b}, call raw_calculate_broken once "
                    f"with operation multiply, operand1 {_calc_a}, operand2 {_calc_b}."
                ),
            )
        else:
            print("  (Set NB_LIVE_RAW_CALC_CONTRAST=1 for optional raw broken multiply.)")

    _nb8_run(_section_calc())

### §4 CALCULATE_RESULT — expected 17×23 = 391
allow ingress-gate-chain INGRESS_ALLOW_DEFAULT | Turn allowed by ingress gate chain.
  gov: tool_progress {'call_id': 'tc_live_calc_planned', 'tool_name': 'calculate_result', 'state': 'queued', 'tool_status': '', 'error_code': '', 'job_id': '', 'lease_token': '', 'lease_expires_at_epoch': '', 'claim_attempt': ''}
  gov: tool_progress {'call_id': 'tc_live_calc_planned', 'tool_name': 'calculate_result', 'state': 'running', 'tool_status': '', 'error_code': '', 'job_id': '', 'lease_token': '', 'lease_expires_at_epoch': '', 'claim_attempt': ''}
  gov: tool_progress {'call_id': 'tc_live_calc_planned', 'tool_name': 'calculate_result', 'state': 'completed', 'tool_status': 'success', 'error_code': 'None', 'job_id': '', 'lease_token': '', 'lease_expires_at_epoch': '', 'claim_attempt': ''}
  gov: output_delta {'text': 'The result of multiplying 17 by 23 is 391.'}
  gov: run_complete {'status': 'completed', 'output': 'The result of multiplying 17 by 2

### §5 Optional model-driven governed turns (diagnostic)

**Off by default** (`NB_LIVE_MODEL_DRIVEN=0`). Natural-language prompts on the OpenAI delegating path — often
**no `TOOL_INTENT`** on the orchestrator stream even when tools run. Compare to **§2–§4** `planned_tool_call` proofs.

In [8]:
if not HAS_OPENAI_KEY:
    print("Skip §5 — run live setup with OPENAI_API_KEY.")
elif not LIVE_MODEL_DRIVEN:
    print("§5 skipped — set NB_LIVE_MODEL_DRIVEN=1 to run model-initiated diagnostic turns.")
else:
    async def _section_model_driven() -> None:
        print("### Model-driven diagnostic (not used for §2–§4 PASS/FAIL)")
        _math_a, _math_b = _nb_live_math_operands()
        p_policy = "MANDATORY: call admin_reset now with no arguments."
        p_math = f"Call safe_add_proven once with a={_math_a} b={_math_b}. Quote sum and proof_token from JSON."
        p_calc = "Call calculate_result for 17 multiply 23. Reply with result field only."

        t_pol = await _governed_turn("sess_nb_md_pol", p_policy, run_label="md_pol")
        il = _nb_turn_str_list(t_pol, "tool_intents")
        bl = _nb_turn_str_list(t_pol, "policy_blocked")
        _nb_verify_line(
            "admin_reset" in il and "admin_reset" in bl,
            "model-driven admin_reset intent + POLICY_BLOCKED (informational)",
            level="WARN",
        )

        t_math = await _governed_turn("sess_nb_md_math", p_math, run_label="md_math")
        cl = _nb_turn_str_list(t_math, "tools_completed")
        _nb_verify_line("safe_add_proven" in cl, "model-driven safe_add_proven completed (informational)", level="WARN")

        t_calc = await _governed_turn("sess_nb_md_calc", p_calc, run_label="md_calc")
        cl2 = _nb_turn_str_list(t_calc, "tools_completed")
        _nb_verify_line("calculate_result" in cl2, "model-driven calculate_result completed (informational)", level="WARN")

    _nb8_run(_section_model_driven())

§5 skipped — set NB_LIVE_MODEL_DRIVEN=1 to run model-initiated diagnostic turns.


### §6 Live governed summary

Run after §1–§4 (and optional §5). Prints one line per section from `_live_gov_summary`.

In [9]:
print("### Live governed summary")
print("tutorial_08 Parts 1–7 are ground truth; below is what optional live cells recorded:")
if _live_gov_summary:
    for sec, msg in _live_gov_summary.items():
        print(f"  {sec}: {msg}")
else:
    print("  (empty — run §1–§4 or enable NB_LIVE_* flags)")
print(
    "\nInterpretation: §1–§4 governed proofs use orchestrator + planned_tool_call (tutorial_08 Part 7). "
    "Raw SDK blocks are ungoverned contrasts only."
)
print("tutorial_09 complete.")

### Live governed summary
tutorial_08 Parts 1–7 are ground truth; below is what optional live cells recorded:
  §1 ingress: PASS — governed path stopped at ingress (pre-model)
  §2 admin_reset policy: FAIL — orchestrator policy proof did not complete
  §3 safe_add_proven: PASS — tool completed + kernel sum/token in reply
  §4 calculate_result: PASS — tool completed + product in reply

Interpretation: §1–§4 governed proofs use orchestrator + planned_tool_call (tutorial_08 Part 7). Raw SDK blocks are ungoverned contrasts only.
tutorial_09 complete.


## Summary — live takeaways

| Section | Story in one line | What to notice |
|---------|-------------------|----------------|
| §1 | Ingress before model spend | Governed stop vs raw SDK on secret text |
| §2 | Tenant overlay blocks tool | `planned_tool_call` → **POLICY_BLOCKED** on `admin_reset` |
| §3 | Deterministic proof tool | Completed tool + kernel **sum** / **proof_token** in reply |
| §4 | Registry parity calc | **17×23** product from tool JSON |
| §5 | Model-driven (optional) | Diagnostic only — not PASS/FAIL for integrators |

Local ground truth: **`tutorial_08`** Parts 1–7. Canonical production ordering:
`docs/architecture/governed-execution-pipeline.md`.

## Notebook navigation

| If you want… | Open |
|---|---|
| Previous / next in learning path | See `notebooks/README.md` index |
| Fast module smoke after a code change | `check_01` … `check_04` |
| Ingress or tool boundary proofs | `edge_01`, `edge_02` |
| Local governance lab (no API key) | `tutorial_08_governed_execution_sandbox.ipynb` |
| Live governance contrasts (optional API key) | `tutorial_09_governed_execution_live.ipynb` |
| Evaluator time-boxed paths | `notebooks/EVALUATOR_GUIDE.md` |

**Regenerate notebooks:** edit this build script, then `python notebooks/build_tutorials.py` (do not hand-edit `.ipynb` JSON).